# Revision v2 GPU Matrix Runner

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/attabeezy/akan-bpe/blob/main/notebooks/revision_gpu_matrix.ipynb)

Runs the frozen 15-run `config/revision_gpu_matrix.yaml` matrix (QLoRA fine-tunes of
Qwen3-0.6B/1.7B under the replacement vs. extension tokenizer strategies, 3 seeds each).

**Designed to be re-run across multiple Kaggle sessions.** Each session works through as
many pending runs as fit in the wall-clock budget below, then stops cleanly before Kaggle's
session limit would kill it mid-run. Attach the `akan-bpe-revision-v2-results` dataset (once
it exists, after the first session) as a second input so completed runs are skipped.

Required inputs (Kaggle "Add Data"):
- `attabeezy/akan-bpe-revision-v2-data` — the frozen `pristine_twi_{train,test}.jsonl` files
  (not committed to git; see `.gitignore`).
- `attabeezy/akan-bpe-revision-v2-results` — optional, prior sessions' completed result JSONs.

**Use the T4 GPU accelerator, not P100** — the preinstalled PyTorch build on Kaggle's
image only supports CUDA capability sm_70+, and the P100 (sm_60) is not compatible with it.
Enable internet access too.

## Setup

In [ ]:
# Clone the repo (skip if already inside it)
import os
from pathlib import Path

REPO = "https://github.com/attabeezy/akan-bpe.git"
REPO_NAME = "akan-bpe"

if Path.cwd().name != REPO_NAME:
    if not Path(REPO_NAME).is_dir():
        !git clone {REPO}
    %cd {REPO_NAME}

print(f"Working directory: {Path.cwd()}")

In [ ]:
# Install runtime dependencies directly (not `pip install -e .`): pyproject.toml
# pins requires-python to 3.13, but Kaggle's base image ships 3.12, so an editable
# install of the akan-bpe package itself fails dependency resolution. We don't need
# the package installed -- the later cell adds the repo root to sys.path instead.
import subprocess
import sys

packages = [
    "datasets>=2.18.0",
    "transformers>=4.51.0",
    "tokenizers>=0.15.0",
    "scikit-learn>=1.4.0",
    "numpy>=1.26.0",
    "protobuf>=5.29.0",
    "PyYAML>=6.0",
    "python-dotenv>=1.0.0",
    "sentencepiece>=0.1.99",
    "tqdm>=4.66.0",
    "accelerate>=0.30.0",
    "peft>=0.11.0",
    "bitsandbytes>=0.43.0",
    "sacrebleu>=2.4.0",
]
result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages])
if result.returncode != 0:
    raise RuntimeError(f"pip install failed with exit code {result.returncode}")
print("Dependencies installed.")

In [ ]:
# Hugging Face authentication (optional: Qwen3 base models are public).
# Add a Kaggle secret named HF_TOKEN (Notebook settings → Secrets) to raise HF's
# rate limits. Never falls back to an interactive prompt -- this notebook is meant
# to run unattended as a Kaggle commit, where a blocking prompt would hang forever.
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face Hub.")
except Exception as exc:
    print(f"No HF_TOKEN secret available ({exc}); continuing unauthenticated.")

In [ ]:
# Confirm a GPU is available
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable a GPU accelerator, then re-run.")
print(f"CUDA devices: {torch.cuda.device_count()}")
for index in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(index)
    print(f"GPU {index}: {props.name} ({props.total_memory / 1e9:.1f} GB)")

In [ ]:
# Stage the frozen data files (gitignored, provided via Kaggle Dataset input)
# and any already-completed results from a prior session, so this session
# resumes instead of re-running finished arms.
import shutil

DATA_INPUT = Path("/kaggle/input/akan-bpe-revision-v2-data")
RESULTS_INPUT = Path("/kaggle/input/akan-bpe-revision-v2-results")

Path("data").mkdir(exist_ok=True)
for name in ("pristine_twi_train.jsonl", "pristine_twi_test.jsonl"):
    src = DATA_INPUT / name
    if not src.exists():
        raise FileNotFoundError(
            f"Missing {src}. Attach the akan-bpe-revision-v2-data dataset as an input."
        )
    shutil.copy(src, Path("data") / name)
    print(f"Staged {name} ({src.stat().st_size} bytes)")

results_dir = Path("results/revision_v2/gpu_runs")
results_dir.mkdir(parents=True, exist_ok=True)
if RESULTS_INPUT.is_dir():
    carried = 0
    for path in RESULTS_INPUT.glob("*.json"):
        shutil.copy(path, results_dir / path.name)
        carried += 1
    print(f"Carried forward {carried} completed result(s) from a prior session.")
else:
    print("No prior-results dataset attached; starting from zero completed runs.")

In [ ]:
!python scripts/run_revision_gpu_matrix.py validate
!python scripts/run_revision_gpu_matrix.py status

## Time-budgeted matrix loop

Runs `run --next` in-process (no subprocess restart cost) until either the matrix is
complete or the wall-clock budget is exhausted. Stops *before* starting a run it doesn't
have time to finish, rather than risking Kaggle killing it mid-run. Any exception in a run
stops the loop immediately (rather than silently skipping) so failures get investigated.

In [ ]:
import sys
import time
import traceback

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from akan_bpe.io import write_json
from akan_bpe.model_integration import run_model_integration
from akan_bpe.revision_gpu import (
    load_revision_matrix,
    model_config_for_run,
    validate_run_result,
)

# Kaggle GPU sessions cap out well under 12h; leave a solid safety margin so a
# save/commit at the end always has time to finish.
TIME_BUDGET_SECONDS = 8.0 * 3600
# Conservative per-run ceiling (the largest arm, qwen-1.7b, is the slow case) used
# to decide whether there's time left to *start* another run.
ASSUMED_MAX_RUN_SECONDS = 150 * 60

matrix = load_revision_matrix(Path("config/revision_gpu_matrix.yaml"))
start = time.monotonic()
completed_this_session = []

while True:
    elapsed = time.monotonic() - start
    remaining = TIME_BUDGET_SECONDS - elapsed
    pending = [run for run in matrix.runs if not run.result_path.exists()]
    if not pending:
        print("All 15 runs complete.")
        break
    if remaining < ASSUMED_MAX_RUN_SECONDS:
        print(
            f"Stopping: only {remaining/60:.1f} min left in budget, "
            f"not enough to safely start another run. {len(pending)} run(s) still pending."
        )
        break

    run = pending[0]
    print(f"\n{'='*80}\nStarting {run.run_id} ({remaining/60:.1f} min left in budget)\n{'='*80}")
    run_start = time.monotonic()
    try:
        config = model_config_for_run(matrix, run)
        payload = run_model_integration(config)
        payload["revision_matrix"] = {
            "matrix_id": matrix.payload["matrix_id"],
            "sha256": matrix.sha256,
            "source": str(matrix.path).replace("\\", "/"),
            "goals": list(run.goals),
        }
        validate_run_result(matrix, run, payload)
        write_json(run.result_path, payload)
        completed_this_session.append(run.run_id)
        print(f"Completed and validated: {run.run_id} ({(time.monotonic()-run_start)/60:.1f} min)")
    except Exception:
        print(f"FAILED: {run.run_id}")
        traceback.print_exc()
        break

print(f"\nCompleted this session: {completed_this_session}")

## Save results for the next session

Zips just the result JSONs (small; checkpoints are not needed after `reload_verification`
already ran in-process) into `/kaggle/working/outputs` so they show up as this notebook
version's output — download them and re-upload as a new version of the
`akan-bpe-revision-v2-results` dataset before the next session.

In [ ]:
!python scripts/run_revision_gpu_matrix.py status
!mkdir -p outputs
!zip -r -j outputs/revision_v2_gpu_results.zip results/revision_v2/gpu_runs
!ls -lh outputs